In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Library

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
tqdm.pandas()


import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Input, Flatten, Dense, Embedding, Dropout, Concatenate
from tensorflow.keras.optimizers import Adam

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

# Utils

In [ ]:
def data_split(df, test_size=0.2, valid_size=0.125, random_state=42):

    # 1차 분할: train+valid, test
    df_train, df_test = train_test_split(df, test_size=test_size, random_state=random_state)

    # 2차 분할: train, valid
    df_train, df_valid = train_test_split(df_train, test_size=valid_size, random_state=random_state)

    # reset index, 혹시나, 없어도 무관
    df_train.reset_index(drop=True, inplace=True)
    df_valid.reset_index(drop=True, inplace=True)
    df_test.reset_index(drop=True, inplace=True)

    'Training Dataset'
    user_id_train = df_train['user_id'].values # 사용자 ID
    item_id_train = df_train['asin'].values # 아이템 ID
    user_bert_train = np.array(df_train['user_bert_emb'].tolist()) # 사용자 BERT
    item_bert_train = np.array(df_train['item_bert_emb'].tolist()) # 사용자 RoBERTa
    user_roberta_train = np.array(df_train['user_roberta_emb'].tolist()) # 아이템 BERT
    item_roberta_train = np.array(df_train['item_roberta_emb'].tolist()) # 아이템 RoBERTa
    y_train = df_train['rating'].values # 평점

    'Valid Dataset'
    user_id_valid = df_valid['user_id'].values # 사용자 ID
    item_id_valid = df_valid['asin'].values # 아이템 ID
    user_bert_valid = np.array(df_valid['user_bert_emb'].tolist()) # 사용자 BERT
    item_bert_valid = np.array(df_valid['item_bert_emb'].tolist()) # 사용자 RoBERTa
    user_roberta_valid = np.array(df_valid['user_roberta_emb'].tolist()) # 아이템 BERT
    item_roberta_valid = np.array(df_valid['item_roberta_emb'].tolist()) # 아이템 RoBERTa
    y_valid = df_valid['rating'].values # 평점

    'Test Dataset'
    user_id_test = df_test['user_id'].values # 사용자 ID
    item_id_test = df_test['asin'].values # 아이템 ID
    user_bert_test = np.array(df_test['user_bert_emb'].tolist()) # 사용자 BERT
    item_bert_test = np.array(df_test['item_bert_emb'].tolist()) # 사용자 RoBERTa
    user_roberta_test = np.array(df_test['user_roberta_emb'].tolist()) # 아이템 BERT
    item_roberta_test = np.array(df_test['item_roberta_emb'].tolist()) # 아이템 RoBERTa
    y_test = df_test['rating'].values # 평점

    return (
        user_id_train, item_id_train, user_bert_train, item_bert_train, user_roberta_train, item_roberta_train, y_train,
        user_id_valid, item_id_valid, user_bert_valid, item_bert_valid, user_roberta_valid, item_roberta_valid, y_valid,
        user_id_test, item_id_test, user_bert_test, item_bert_test, user_roberta_test, item_roberta_test, y_test
    )

# Load Data

In [ ]:
# 데이터 로딩
df = pd.read_pickle('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Clothing_Shoes_v4.pkl')

print(df.shape)
df.head()

(66718, 15)


,user_id,asin,parent_asin,text,rating,title_y,roberta_emb,small_image_url,medium_image_url,large_image_url,VGG16_emb,user_bert_emb,item_bert_emb,user_roberta_emb,item_roberta_emb
0,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B08HQPWVCC,B08HQPWVCC,Pretty colored earrings but color of stainless...,4,Udalyn 6pairs Stud Earrings Set for Women Opal...,"[-0.08417237550000001, 0.1606151462, -0.014070...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"[0.0, 12.1029911041, 0.0, 15.7332105637, 4.268...","[-0.5218123, -0.96434146, 0.4619303, -0.409749...","[-0.20688048, -0.32638466, 0.13180715, -0.5702...","[-0.110578634, 0.11269227, -0.041261047, -0.10...","[-0.06857138, 0.13921358, -0.039587017, -0.088..."
1,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B08JQ88186,B08JQ88186,Great little earrings. Perfect size. Not too b...,5,ZENI 925 Sterling Silver Snowflake Earrings fo...,"[-0.0825007036, 0.175420925, -0.054644376, -0....",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"[0.0, 0.0, 2.2863385677, 10.4048433304, 6.9813...","[-0.5218123, -0.96434146, 0.4619303, -0.409749...","[0.062450625, -0.07075825, 0.15572476, -0.2797...","[-0.110578634, 0.11269227, -0.041261047, -0.10...","[-0.04617965, 0.12618569, -0.017711096, -0.102..."
2,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07YXXR1DG,B07YXXR1DG,Great bracelet key chain. Perfect for busy mom...,5,Bangle key ring Bracelet Keychain wristlet key...,"[-0.09092549230000001, 0.1620114595, -0.038774...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"[0.0, 37.5705032349, 2.7596354485, 0.0, 12.911...","[-0.5218123, -0.96434146, 0.4619303, -0.409749...","[-0.26902932, -0.42407188, 0.82339007, -0.2940...","[-0.110578634, 0.11269227, -0.041261047, -0.10...","[-0.062135138, 0.1333665, -0.019405615, -0.102..."
3,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07RHSJQQN,B07RHSJQQN,This is a nice patriotic tank. The medium fits...,4,YEXIPO Womens American Flag Racerback Tank Top...,"[-0.12041848150000001, 0.0892145857, -0.027717...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"[0.0, 18.6175479889, 0.0, 0.0, 3.5440309048, 0...","[-0.5218123, -0.96434146, 0.4619303, -0.409749...","[0.119323395, -0.57359934, 0.40760195, -0.2408...","[-0.110578634, 0.11269227, -0.041261047, -0.10...","[-0.06780286, 0.051634714, -0.017799914, -0.11..."
4,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07LFZRYC2,B07LFZRYC2,Great boys shirt. My grandson wears a medium i...,5,Hurley Boys' One and Ony Graphic T-Shirt,"[-0.1369395405, 0.07035727800000001, -0.060328...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,"[1.9380488396, 0.0, 0.0, 6.4230856895, 0.0, 0....","[-0.5218123, -0.96434146, 0.4619303, -0.409749...","[-0.43734312, -0.62041384, 0.14694075, -0.3723...","[-0.110578634, 0.11269227, -0.041261047, -0.10...","[-0.07941287, 0.05176169, -0.02554227, -0.0903..."


In [ ]:
user_encoder = LabelEncoder()
item_encoder = LabelEncoder()

df['user_id'] = user_encoder.fit_transform(df['user_id'])
df['asin'] = item_encoder.fit_transform(df['asin'])

# 고유한 user와 item 수를 집계
n_users = df['user_id'].nunique()
print(n_users)
n_items = df['asin'].nunique()
print(n_items)

7449
16099


In [ ]:
user_id_train, item_id_train, user_bert_train, item_bert_train, user_roberta_train, item_roberta_train, y_train, \
user_id_valid, item_id_valid, user_bert_valid, item_bert_valid, user_roberta_valid, item_roberta_valid, y_valid, \
user_id_test, item_id_test, user_bert_test, item_bert_test, user_roberta_test, item_roberta_test, y_test = data_split(df)

# Model

In [ ]:
def MFNR(num_users, num_items, dropout, learning_rate):

    # 사용자 ID
    user_id_input = Input(shape=(1,), name='user_id_input')
    user_emb = Embedding(input_dim=num_users, output_dim=64, name='user_embedding')(user_id_input)
    user_emb = Flatten()(user_emb)

    # 아이템 ID
    item_id_input = Input(shape=(1,), name='item_id_input')
    item_emb = Embedding(input_dim=num_items, output_dim=64, name='item_embedding')(item_id_input)
    item_emb = Flatten()(item_emb)

    # 사용자 멀티 임베딩
    user_bert_input = Input(shape=(768,), name='user_bert_input')
    user_roberta_input = Input(shape=(768,), name='user_roberta_input')

    # 아이템 멀티 임베딩
    item_bert_input = Input(shape=(768,), name='item_bert_input')
    item_roberta_input = Input(shape=(768,), name='item_roberta_input')

    # 사용자 리뷰 MLP
    user_reviews = Concatenate()([user_bert_input, user_roberta_input])

    user_nlp = Dense(512, activation='relu')(user_reviews)
    user_nlp = Dropout(dropout)(user_nlp)
    user_nlp = Dense(256, activation='relu')(user_nlp)
    user_nlp = Dropout(dropout)(user_nlp)
    user_nlp = Dense(128, activation='relu')(user_nlp)
    user_nlp = Dropout(dropout)(user_nlp)

    # 사용자 결합
    user_rep = Concatenate()([user_emb, user_nlp])

    # 아이템 리뷰 MLP
    item_reviews = Concatenate()([item_bert_input, item_roberta_input])

    item_nlp = Dense(512, activation='relu')(item_reviews)
    item_nlp = Dropout(dropout)(item_nlp)
    item_nlp = Dense(256, activation='relu')(item_nlp)
    item_nlp = Dropout(dropout)(item_nlp)
    item_nlp = Dense(128, activation='relu')(item_nlp)
    item_nlp = Dropout(dropout)(item_nlp)

    # 아이템 결합
    item_rep = Concatenate()([item_emb, item_nlp])

    # 예측
    user_item_interaction = Concatenate(name='all_representation')([user_rep, item_rep])
    prediction = Dense(64, activation='relu')(user_item_interaction)
    prediction = Dropout(dropout)(prediction)
    prediction = Dense(32, activation='relu')(prediction)
    prediction = Dropout(dropout)(prediction)
    prediction = Dense(16, activation='relu')(prediction)
    prediction = Dropout(dropout)(prediction)

    final = Dense(1, activation='linear', name='rating_prediction')(prediction)

    model = Model(inputs=[user_id_input, item_id_input, user_bert_input, user_roberta_input, item_bert_input, item_roberta_input],
                  outputs=final)
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='mean_squared_error', metrics=['mean_absolute_error', 'mean_squared_error'])

    return model

In [ ]:
dropout = 0.0
learning_rate = 0.0001

In [ ]:
model = MFNR(n_users, n_items, dropout, learning_rate)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ user_bert_input     │ (None, 768)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_roberta_input  │ (None, 768)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ item_bert_input     │ (None, 768)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ item_roberta_input  │ (None, 768)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 1536)      │          0 │ user_bert_input[… │
│ (Concatenate)       │                   │            │ user_roberta_inp… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_2       │ (None, 1536)      │          0 │ item_bert_input[… │
│ (Concatenate)       │                   │            │ item_roberta_inp… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 512)       │    786,944 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 512)       │    786,944 │ concatenate_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 512)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 512)       │          0 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 256)       │    131,328 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 256)       │    131,328 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_id_input       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 256)       │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ item_id_input       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 256)       │          0 │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_embedding      │ (None, 1, 64)     │    476,736 │ user_id_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 128)       │     32,896 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ item_embedding      │ (None, 1, 64)     │  1,030,336 │ item_id_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 128)       │     32,896 │ dropout_4[0][0] 

 Total params: 3,436,673 (13.11 MB)

 Trainable params: 3,436,673 (13.11 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
early_stopping = EarlyStopping(monitor ='val_loss', min_delta = 0.001, patience = 5, verbose = 1, mode = 'min', restore_best_weights = True)

In [ ]:
history = model.fit([user_id_train, item_id_train, user_bert_train, item_bert_train, user_roberta_train, item_roberta_train],
                    y_train,
                    validation_data = ([user_id_valid, item_id_valid, user_bert_valid, item_bert_valid, user_roberta_valid, item_roberta_valid], y_valid),
                    epochs = 100,
                    batch_size = 128,
                    callbacks=[early_stopping])

Epoch 1/100
365/365 ━━━━━━━━━━━━━━━━━━━━ 32s 46ms/step - loss: 2.8441 - mean_absolute_error: 1.1139 - mean_squared_error: 2.8441 - val_loss: 0.4759 - val_mean_absolute_error: 0.5200 - val_mean_squared_error: 0.4759
Epoch 2/100
365/365 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.4809 - mean_absolute_error: 0.5126 - mean_squared_error: 0.4809 - val_loss: 0.4599 - val_mean_absolute_error: 0.4992 - val_mean_squared_error: 0.4599
Epoch 3/100
365/365 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.4522 - mean_absolute_error: 0.4933 - mean_squared_error: 0.4522 - val_loss: 0.4524 - val_mean_absolute_error: 0.5117 - val_mean_squared_error: 0.4524
Epoch 4/100
365/365 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.4217 - mean_absolute_error: 0.4686 - mean_squared_error: 0.4217 - val_loss: 0.4401 - val_mean_absolute_error: 0.4882 - val_mean_squared_error: 0.4401
Epoch 5/100
365/365 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.3587 - mean_absolute_error: 0.4204 - mean_squared_error: 0.3587 - val_loss: 0.4473 

In [ ]:
# 모델 성능 측정
predicted_ratings = model.predict([user_id_test, item_id_test, user_bert_test, item_bert_test, user_roberta_test, item_roberta_test])

MAE = mean_absolute_error(y_test, predicted_ratings)
MSE = mean_squared_error(y_test, predicted_ratings)
RMSE = np.sqrt(MSE)
MAPE = mean_absolute_percentage_error(y_test, predicted_ratings) * 100

print(f'MAE: {MAE:.4f}')
print(f'MSE: {MSE:.4f}')
print(f'RMSE: {RMSE:.4f}')
print(f'MAPE: {MAPE:.4f}%')

417/417 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step
MAE: 0.4911
MSE: 0.4483
RMSE: 0.6696
MAPE: 14.1563%
